# Illustration of a Diffusion Process

Antonio Esteves @UMinho, February 2025

In [ ]:
import numpy             as     np
from   scipy.stats       import norm, uniform
import matplotlib.pyplot as     plt
import seaborn           as     sns

## A Uniform distribution

In [ ]:
x = np.linspace(-6, +10, 200)

u = []
umean   = -2.0 # minimum
ustddev = 5.0  # width
for xi in x:
    ui = uniform.pdf(xi,loc=umean, scale=ustddev)
    u.append(ui)

fig, ax = plt.subplots(1, 1)
ax.plot(x, u, 'b-', lw=5, alpha=0.6, label=f'Uinforn({umean},{ustddev}) pdf')
ax.set_title('Uniform(-2, 5)')

The histogram of $10^4$ samples drawn from the uniform distribution. 

In [ ]:
umean   = -2.0 # minimum
ustddev = 5.0  # width
u       = uniform.rvs(loc=umean, scale=ustddev, size=10000)

fig, ax = plt.subplots(1, 1)

ax.hist(u, density=True, bins='auto', histtype='stepfilled', alpha=0.2)
ax.set_xlim(-6, 10)
ax.legend(['u'], loc='best', frameon=False)
plt.show()

## Generate a distribution $y$ as a mixture of two Gaussian

The first Gaussian is N(-2.5,1.0) and the second is N(3.0,2.0).
The first Gaussian is samples 60% of the time and the second 40%.

In [ ]:
x = np.linspace(-6, +10, 200)
y = []
p1 = 0.6
p2 = 0.4
for xi in x:
    yi = p1 * norm.pdf(xi, loc=-2.5, scale=1.0) + p2 * norm.pdf(xi, loc=3.0, scale=2.0)
    y.append(yi)

fig, ax = plt.subplots(1, 1)
ax.plot(x, y, 'r-', lw=5, alpha=0.6, label='X pdf')
ax.set_title('pdf(y)')


### Plot the histogram of $10^4$ samples drawn from the created distribution $y$

In [ ]:
x     = []
limit = 0.6
min   = 0.0
max   = 1.0

for p in range(10000):
    rnd = uniform.rvs(loc=min, scale=max, size=1)
    if rnd < limit:
        xi = norm.rvs(loc=-2.5, scale=1.0, size=1)
    else:
        xi = norm.rvs(loc=3, scale=2.0, size=1)
    x.append(xi[0])

sns.histplot(
    x,
    kde=True,
    stat='density',
    bins=50,
    color = 'red', 
    )

## Apply the diffusion process to the created distribution $y$

Here we apply the diffusion process to 8 samples from the distribution $y$ using the stochastic differential equation $y_t = \sqrt{1-p} y_{t-1} + \sqrt{p} u_t$, where $u_t \sim \mathcal{N}(0,1)$ and $p=0.01$.

We plot the trajectory of these 8 samples during 300 steps.

In [ ]:
p              = 0.01
sqrt_p         = np.sqrt(p)
sqrt_1_minus_p = np.sqrt(1.0-p)
limit          = 0.6
min            = 0.0
max            = 1.0
points         = 8
steps          = 300

xt = np.ndarray(shape=(points,steps), dtype=float)

for p in range(points):
    rnd = uniform.rvs(loc=min, scale=max, size=1)
    if rnd < limit:
        x0 = norm.rvs(loc=-2.5, scale=1.0, size=1)
    else:
        x0 = norm.rvs(loc=3, scale=2.0, size=1)
    xt[p][0] = x0[0]

    for s in range(1,steps):
        ut = norm.rvs(loc=0, scale=1, size=1)[0]
        xt[p][s] = sqrt_1_minus_p * xt[p][s-1] + sqrt_p * ut

fig, ax = plt.subplots(1, 1)

for p in range(points):
    ax.plot(xt[p])
    ax.set_xlim(0, steps)
ax.legend(['x'], loc='best', frameon=False)
plt.show()


Here we apply the diffusion process to $10^4$ samples from the distribution $y$ using the stochastic differential equation $y_t = \sqrt{1-p} y_{t-1} + \sqrt{p} u_t$, where $u_t \sim \mathcal{N}(0,1)$ and $p=0.01$.

We plot the histogram of the samples after 300 steps and we confirm that it is becoming close to a standard normal distribution.

In [ ]:
p              = 0.01
sqrt_p         = np.sqrt(p)
sqrt_1_minus_p = np.sqrt(1.0-p)
limit          = 0.6
min            = 0.0
max            = 1.0
points         = 10000
steps          = 300

xt = np.ndarray(shape=(points,steps), dtype=float)

for p in range(points):
    rnd = uniform.rvs(loc=min, scale=max, size=1)
    if rnd < limit:
        x0 = norm.rvs(loc=-2.5, scale=1.0, size=1)
    else:
        x0 = norm.rvs(loc=3, scale=2.0, size=1)
    xt[p][0] = x0[0]

    for s in range(1,steps):
        ut = norm.rvs(loc=0, scale=1, size=1)[0]
        xt[p][s] = sqrt_1_minus_p * xt[p][s-1] + sqrt_p * ut

x_final = np.ndarray(shape=(points), dtype=float)

for p in range(points):
    x_final[p] = xt[p][-1]

sns.histplot(
    x_final,
    kde=True,
    stat='density',
    bins=50,
    color = 'skyblue', 
    )

In [ ]:
sns.kdeplot(
    x_final,
    color = 'darkblue', 
    )